# Data Analysis

In [35]:
import duckdb
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [36]:
# Setup logger

logging.basicConfig(
    filename='./logs/data_analysis.log',
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='w'
)
logger = logging.getLogger(__name__)

## Database Querying

In [37]:
# Query the database

try:
    
    # Establish DuckDB connection
    con = duckdb.connect(database='./sp500.duckdb', read_only=False)
    logger.info("Connected to DuckDB instance")

    # Query the sp500 Database
    company_table = con.execute("""
        SELECT
            c.Symbol, c.Shortname, c.Sector, c.Industry,
            e.totalEsg, e.environmentScore, e.socialScore, e.governanceScore, e.percentile, e.beta, e.highestControversy, e.overallRisk,
            AVG(s.Open) AS avg_open, AVG(s.Close) AS avg_close, AVG(s.High) AS avg_high, AVG(s.Low) AS avg_low, AVG(s.Volume) AS avg_volume, AVG((s.Close - s.Open) / s.Open) AS avg_daily_return, STDDEV(s.Close) AS price_volatility
        FROM companies c
        LEFT JOIN esg e
            ON c.Symbol = e.Symbol
        LEFT JOIN stocks s
            ON c.Symbol = s.Symbol
        INNER JOIN constituents cn
            ON c.Symbol = cn.Symbol
        GROUP BY
            c.Symbol, c.Shortname, c.Sector, c.Industry,
            e.totalEsg, e.environmentScore, e.socialScore, e.governanceScore, e.percentile, e.beta, e.highestControversy, e.overallRisk
        ORDER BY avg_close DESC;
    """).df()
    logger.info("Query complete")

    # Close DuckDB connection
    con.close()
    logger.info("Closed DuckDB connection")

except Exception as e:
    print(f"An error occurred: {e}")
    logger.error(f"An error occurred: {e}")

In [38]:
company_table.head()


,Symbol,Shortname,Sector,Industry,totalEsg,environmentScore,socialScore,governanceScore,percentile,beta,highestControversy,overallRisk,avg_open,avg_close,avg_high,avg_low,avg_volume,avg_daily_return,price_volatility
0,MTD,"Mettler-Toledo International, I",Healthcare,Diagnostics & Research,13.10,1.00,7.41,4.69,6.93,1.139,0.0,5,644.892789,645.048911,652.233365,637.424992,1.623200e+05,0.000650,469.949198
1,EQIX,"Equinix, Inc.",Real Estate,REIT - Specialty,13.88,3.44,5.31,5.13,8.50,0.700,1.0,5,426.004031,426.076441,430.665350,421.111144,6.753230e+05,0.000431,254.737572
2,TDG,Transdigm Group Incorporated,Industrials,Aerospace & Defense,38.72,12.01,17.71,9.00,90.45,1.410,2.0,10,394.221516,394.298474,398.558856,389.727444,4.060174e+05,0.000629,318.809429
3,NFLX,"Netflix, Inc.",Communication Services,Entertainment,16.41,0.09,7.31,9.01,15.33,1.262,2.0,9,232.776050,232.850524,236.217270,229.286021,1.664855e+07,0.000911,210.464293
4,FDS,FactSet Research Systems Inc.,Financial Services,Financial Data & Stock Exchanges,18.37,4.32,8.75,5.30,21.85,0.748,1.0,6,226.933893,227.038344,229.227513,224.720395,3.067631e+05,0.000656,129.810908


In [39]:
company_table.isna().sum()

Symbol                  0
Shortname               0
Sector                  0
Industry                0
totalEsg               69
environmentScore       69
socialScore            69
governanceScore        69
percentile             69
beta                   69
highestControversy     69
overallRisk            69
avg_open              310
avg_close             310
avg_high              310
avg_low               310
avg_volume            310
avg_daily_return      310
price_volatility      310
dtype: int64

In [45]:
# Data cleaning - drop NAs

company_table = company_table.dropna()

new_len = len(company_table)
print("Rows:", new_len)

Rows: 142


The stocks table has lots of missing fields; the output company table has 66% NA values for the stock fields. Dropping the NAs hurts this analysis but does not prevent the development of a model. The cleaned table has 142 entries.

## Modeling